# GFS要素清单：1-5

In [ ]:
from reki import from_source

In [ ]:
from data_notebook.data import get_data_file

gfs_grib2_orig_file_path = get_data_file("cma-gfs")

examples = {}
gfs_grib2_orig_file_path

## ACPCP/NCPCP/APCP/ASNOW

### 对流降水

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 对流降水 | rainc | ACPCP | acpcp | (0, 1, 10) | - | - | 1 |

In [ ]:
examples[1] = (
    {"parameter": "cedarkit.rainc.since-initialization"},
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 10,
        "productDefinitionTemplateNumber": 8,
        "typeOfFirstFixedSurface": 1,
        "typeOfSecondFixedSurface": 255,
        "typeOfStatisticalProcessing": 1,
        "forecastTime": 0,
        "lengthOfTimeRange": 24,
    },
)

### 大尺度降水

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 大尺度降水 | rainnc | NCPCP | ncpcp | (0, 1, 9) | - | - | 2 |

In [ ]:
examples[2] = (
    {"parameter": "cedarkit.rainnc.since-initialization"},
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 9,
        "productDefinitionTemplateNumber": 8,
        "typeOfFirstFixedSurface": 1,
        "typeOfSecondFixedSurface": 255,
        "typeOfStatisticalProcessing": 1,
        "forecastTime": 0,
        "lengthOfTimeRange": 24,
    },
)

### 总降水

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 总降水 | rain | APCP | - | (0, 1, 8) | - | - | 3 |

In [ ]:
examples[3] = (
    {"parameter": "cedarkit.rain.since-initialization"},
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 8,
        "productDefinitionTemplateNumber": 8,
        "typeOfFirstFixedSurface": 1,
        "typeOfSecondFixedSurface": 255,
        "typeOfStatisticalProcessing": 1,
        "forecastTime": 0,
        "lengthOfTimeRange": 24,
    },
)

### 降雪量

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 降雪量 | snow | ASNOW | - | (0, 1, 29) | - | - | 4 |

In [ ]:
examples[4] = (
    {"parameter": "cedarkit.sf.large-scale.since-initialization"},
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 29,
        "productDefinitionTemplateNumber": 8,
        "typeOfFirstFixedSurface": 1,
        "typeOfSecondFixedSurface": 255,
        "typeOfStatisticalProcessing": 1,
        "forecastTime": 0,
        "lengthOfTimeRange": 24,
    },
)

## TMP

### 地表温度

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 地表温度 | ts | TMP | t | (0, 0, 0) | surface | - | 5 |

In [ ]:
examples[5] = (
    {"parameter": "cedarkit.ts"},
    {
        "discipline": 0,
        "parameterCategory": 0,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 0,
        "typeOfFirstFixedSurface": 1,
        "typeOfSecondFixedSurface": 255,
    },
)

## 核对具名查询与原生键

In [ ]:
with from_source({"kind": "file", "path": str(gfs_grib2_orig_file_path)}) as source:
    named = source.sel([query for query, _ in examples.values() if query is not None]).all()
    native = source.sel([{"native": {"grib": keys}} for _, keys in examples.values()]).all()
    fields = {field.metadata["source"]["ordinal"]: field for field in named}
    native_fields = {field.metadata["source"]["ordinal"]: field for field in native}
    assert native_fields.keys() == examples.keys()
    assert fields.keys() == {ordinal for ordinal, (query, _) in examples.items() if query is not None}

    for ordinal, field in fields.items():
        by_native = native_fields[ordinal]
        assert field.identity.parameter == examples[ordinal][0]["parameter"]
        assert field.metadata["source"] == by_native.metadata["source"]
        assert field.read_grib_message() == by_native.read_grib_message()

    with gfs_grib2_orig_file_path.open("rb") as original:
        for field in native_fields.values():
            location = field.metadata["source"]
            original.seek(location["offset"])
            assert field.read_grib_message() == original.read(location["length"])